In [1]:
import os
from openai import OpenAI
import json
from pydantic import BaseModel
from typing import List
import rich

In [2]:
API_KEY = os.environ.get('OPENAI_API_KEY')
BASE_URL = os.environ.get('OPENAI_BASE_URL')
MODEL = "gpt-5.4"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Passing json schema or Pydantic object in API call to enforce structured output**

# Chat Completion API

https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat

To enforce structued output we will use `parse` function of API like this `openai.beta.chat.completions.parse`, and pass `response_format` property to send json schema

In [3]:
response = openai.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": "提取事件信息 "},
        {"role": "user", "content": "Alice and Bob 准备周五去参加科学研讨会。"},
    ],
    # response_format property is different in chat and responses api
    response_format={
        # schema format is also bit different in chat and responses api
        "type": "json_schema",
        "json_schema": {
            "name": "calendar_event",
            "schema": {
                "type": "object",
                "properties": {
                    "name": { "type": "string"},
                    "date": { "type": "string" },
                    "participants": { "type": "array", "items": { "type": "string" }},
                },
                "required": ["name", "date", "participants"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)
print(response.choices[0].message.content)
dictionary = json.loads(response.choices[0].message.content)
rich.print(dictionary)
print(dictionary["name"])


print()
# This will return None, but if we provide pydantic's BaseModel object in API call
# then we will get parsed object. See example below
print(response.choices[0].message.parsed)

{"name":"科学研讨会","date":"周五","participants":["Alice","Bob"]}


{'name': '科学研讨会', 'date': '周五', 'participants': ['Alice', 'Bob']}

科学研讨会

None


A similar example, except for the user input.

In [4]:
response = openai.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": "提取事件信息"},
        {"role": "user", "content": "Leonardo, Ivan and Alex 准备周二晚上去 Taylor 家吃晚餐。"},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "calendar_event",
            "schema": {
                "type": "object",
                "properties": {
                    "name": { "type": "string"},
                    "date": { "type": "string" },
                    "participants": { "type": "array", "items": { "type": "string" }},
                },
                "required": ["name", "date", "participants"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)
print(response.choices[0].message.content)
dictionary = json.loads(response.choices[0].message.content)
print(dictionary["name"])

{"name":"去 Taylor 家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}
去 Taylor 家吃晚餐


Defining object schemas using Pydantic

Note: `response.choices[0].message.parsed` directly returns object

In [5]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = openai.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": "提取事件信息"},
        {"role": "user", "content": "Leonardo, Ivan and Alex 准备周二晚上去 Taylor 家吃晚餐。"},
    ],
    response_format=CalendarEvent
)
# We have json string which we can convert into json object
print(response.choices[0].message.content)
dictionary = json.loads(response.choices[0].message.content)
print(dictionary["name"])

print()
# Also 'response.choices[0].message' and 'parsed' property that will return parsed object
print(response.choices[0].message.parsed)
rich.print(response.choices[0].message.parsed)
obj = response.choices[0].message.parsed
print(obj.name)

{"name":"在 Taylor 家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}
在 Taylor 家吃晚餐

name='在 Taylor 家吃晚餐' date='周二晚上' participants=['Leonardo', 'Ivan', 'Alex', 'Taylor']


CalendarEvent(name='在 Taylor 家吃晚餐', date='周二晚上', participants=['Leonardo', 'Ivan', 'Alex', 'Taylor'])

在 Taylor 家吃晚餐


# Responses API

https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses

To enforce structued output we can use same `create` function and pass `text` property to send json schema

In [6]:
response = openai.responses.create(
    model=MODEL,
    input=[
        {"role": "developer", "content": "提取事件信息 "},
        {"role": "user", "content": "Leonardo, Ivan and Alex 准备周二晚上去 Taylor 家吃晚餐。"},
    ],
    # text property is different in chat and responses api
    text={
        # schema format is also bit different in chat and responses api
        "format": {
            "type": "json_schema",
            "name": "calendar_event",
            "schema": {
                "type": "object",
                "properties": {
                    "name": { "type": "string"},
                    "date": { "type": "string" },
                    "participants": { "type": "array", "items": { "type": "string" }},
                },
                "required": ["name", "date", "participants"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)
print(response.output_text)
dictionary = json.loads(response.output_text)
print(dictionary["name"])

{"name":"去 Taylor 家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}
去 Taylor 家吃晚餐


In the example below, we use `openai.responses.parse()`, and the property for format is `text_format` instead of `text`.

https://github.com/openai/openai-python/blob/main/examples/responses/structured_outputs.py

In [7]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = openai.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": "提取事件信息 "},
        {"role": "user", "content": "Leonardo, Ivan and Alex 准备周二晚上去 Taylor 家吃晚餐。"},
    ],
    text_format=CalendarEvent
)
print(response.output_text)
dictionary = json.loads(response.output_text)
print(dictionary["name"])

{"name":"去Taylor家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}
去Taylor家吃晚餐


Similar example, execpt for handling parsed object in response

Note: `response.output[0].content[0].parsed` directly returns object

In [8]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = openai.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": "提取事件信息 "},
        {"role": "user", "content": "Leonardo, Ivan and Alex 准备周二晚上去 Taylor 家吃晚餐。"},
    ],
    text_format=CalendarEvent
)
print("status:", response.status)
print("error:", response.error)
print("incomplete_details:", response.incomplete_details)
print("output_text:", response.output_text)
print("output is None:", response.output is None)
rich.print(response)
print()
if response.output and response.output[-1].content:
  parsed = response.output[-1].content[0].parsed
  print(parsed)
  obj = response.output[-1].content[0].parsed
  print(obj.date)
else:
  print("No output. status=", response.status, "error=", response.error)

status: completed
error: None
incomplete_details: None
output_text: {"name":"去 Taylor 家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}
output is None: False


ParsedResponse[CalendarEvent](
    id='resp_083ef4f52e30b2080169d901740c3881939cf42116d708d57f',
    created_at=1775829364.0,
    error=None,
    incomplete_details=None,
    instructions=None,
    metadata={},
    model='gpt-5.4',
    object='response',
    output=[
        ParsedResponseOutputMessage[CalendarEvent](
            id='msg_083ef4f52e30b2080169d9017567648193b30022d9f538b02b',
            content=[
                ParsedResponseOutputText[CalendarEvent](
                    annotations=[],
                    text='{"name":"去 Taylor 
家吃晚餐","date":"周二晚上","participants":["Leonardo","Ivan","Alex","Taylor"]}',
                    type='output_text',
                    logprobs=[],
                    parsed=CalendarEvent(
                        name='去 Taylor 家吃晚餐',
                        date='周二晚上',
                        participants=['Leonardo', 'Ivan', 'Alex', 'Taylor']
                    )
                )
            ],
            role='assistant',
            status='completed',
            type='message',
            phase='final_answer'
        )
    ],
    parallel_tool_calls=True,
    temperature=1.0,
    tool_choice='auto',
    tools=[],
    top_p=0.98,
    background=False,
    conversation=None,
    max_output_tokens=None,
    max_tool_calls=None,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key='dedfb192-62a8-412d-81cb-f8c67e411fb4',
    reasoning=Reasoning(effort='none', generate_summary=None, summary=None),
    safety_identifier='user-8xExoKtAVrEWwVtkLKy7MA4w',
    service_tier='default',
    status='completed',
    text=ResponseTextConfig(
        format=ResponseFormatTextJSONSchemaConfig(
            name='CalendarEvent',
            schema_={
                'properties': {
                    'name': {'title': 'Name', 'type': 'string'},
                    'date': {'title': 'Date', 'type': 'string'},
                    'participants': {'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}
                },
                'required': ['name', 'date', 'participants'],
                'title': 'CalendarEvent',
                'type': 'object',
                'additionalProperties': False
            },
            type='json_schema',
            description=None,
            strict=True
        ),
        verbosity='medium'
    ),
    top_logprobs=0,
    truncation='disabled',
    usage=ResponseUsage(
        input_tokens=94,
        input_tokens_details=InputTokensDetails(cached_tokens=0),
        output_tokens=36,
        output_tokens_details=OutputTokensDetails(reasoning_tokens=0),
        total_tokens=130
    ),
    user=None,
    completed_at=1775829366,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    prompt_cache_retention=None,
    store=False,
    tool_usage={
        'image_gen': {
            'input_tokens': 0,
            'input_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'output_tokens': 0,
            'output_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'total_tokens': 0
        },
        'web_search': {'num_requests': 0}
    }
)


name='去 Taylor 家吃晚餐' date='周二晚上' participants=['Leonardo', 'Ivan', 'Alex', 'Taylor']
周二晚上


You can ask the model to output an answer in a structured, step-by-step way, to guide the user through the solution.

In [9]:
class Step(BaseModel):
    explanation: str
    output: str


class MathResponse(BaseModel):
    steps: List[Step]
    final_answer: str

response = openai.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": "你是一个乐于助人的数学助教，指导学生要一步一步作答。"},
        {"role": "user", "content": "how can I solve 8x + 7 = -23"},
    ],
    text_format=MathResponse
)

rich.print(response)

ParsedResponse[MathResponse](
    id='resp_07e43e0840bc3b620169d9017761f481959b97d2ca18e8c5e7',
    created_at=1775829367.0,
    error=None,
    incomplete_details=None,
    instructions=None,
    metadata={},
    model='gpt-5.4',
    object='response',
    output=[
        ParsedResponseOutputMessage[MathResponse](
            id='msg_07e43e0840bc3b620169d9017809348195bcdef2afb2734c77',
            content=[
                ParsedResponseOutputText[MathResponse](
                    annotations=[],
                    text='{"steps":[{"explanation":"Start with the equation.","output":"8x + 7 = 
-23"},{"explanation":"Subtract 7 from both sides to isolate the term with x.","output":"8x = 
-30"},{"explanation":"Divide both sides by 8.","output":"x = -30/8"},{"explanation":"Simplify the 
fraction.","output":"x = -15/4"}],"final_answer":"x = -15/4"}',
                    type='output_text',
                    logprobs=[],
                    parsed=MathResponse(
                        steps=[
                            Step(explanation='Start with the equation.', output='8x + 7 = -23'),
                            Step(
                                explanation='Subtract 7 from both sides to isolate the term with x.',
                                output='8x = -30'
                            ),
                            Step(explanation='Divide both sides by 8.', output='x = -30/8'),
                            Step(explanation='Simplify the fraction.', output='x = -15/4')
                        ],
                        final_answer='x = -15/4'
                    )
                )
            ],
            role='assistant',
            status='completed',
            type='message',
            phase='final_answer'
        )
    ],
    parallel_tool_calls=True,
    temperature=1.0,
    tool_choice='auto',
    tools=[],
    top_p=0.98,
    background=False,
    conversation=None,
    max_output_tokens=None,
    max_tool_calls=None,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key='14123a8a-9814-4bfc-a5ff-291a42e57b48',
    reasoning=Reasoning(effort='none', generate_summary=None, summary=None),
    safety_identifier='user-lnWFs7cT5KSV6TEFrgfWiZgL',
    service_tier='default',
    status='completed',
    text=ResponseTextConfig(
        format=ResponseFormatTextJSONSchemaConfig(
            name='MathResponse',
            schema_={
                '$defs': {
                    'Step': {
                        'properties': {
                            'explanation': {'title': 'Explanation', 'type': 'string'},
                            'output': {'title': 'Output', 'type': 'string'}
                        },
                        'required': ['explanation', 'output'],
                        'title': 'Step',
                        'type': 'object',
                        'additionalProperties': False
                    }
                },
                'properties': {
                    'steps': {'items': {'$ref': '#/$defs/Step'}, 'title': 'Steps', 'type': 'array'},
                    'final_answer': {'title': 'Final Answer', 'type': 'string'}
                },
                'required': ['steps', 'final_answer'],
                'title': 'MathResponse',
                'type': 'object',
                'additionalProperties': False
            },
            type='json_schema',
            description=None,
            strict=True
        ),
        verbosity='medium'
    ),
    top_logprobs=0,
    truncation='disabled',
    usage=ResponseUsage(
        input_tokens=137,
        input_tokens_details=InputTokensDetails(cached_tokens=0),
        output_tokens=102,
        output_tokens_details=OutputTokensDetails(reasoning_tokens=0),
        total_tokens=239
    ),
    user=None,
    completed_at=1775829372,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    prompt_cache_retention=None,
    store=False,
    tool_usage={
        'image_gen': {
            'input_tokens': 0,


In [10]:
math_reasoning = response.output[-1].content[0].parsed
print(math_reasoning)
print()
print(math_reasoning.final_answer)
print()
rich.print(math_reasoning)

steps=[Step(explanation='Start with the equation.', output='8x + 7 = -23'), Step(explanation='Subtract 7 from both sides to isolate the term with x.', output='8x = -30'), Step(explanation='Divide both sides by 8.', output='x = -30/8'), Step(explanation='Simplify the fraction.', output='x = -15/4')] final_answer='x = -15/4'

x = -15/4



MathResponse(
    steps=[
        Step(explanation='Start with the equation.', output='8x + 7 = -23'),
        Step(explanation='Subtract 7 from both sides to isolate the term with x.', output='8x = -30'),
        Step(explanation='Divide both sides by 8.', output='x = -30/8'),
        Step(explanation='Simplify the fraction.', output='x = -15/4')
    ],
    final_answer='x = -15/4'
)